# 💵 Currency Note Detector — Debugged & Complete

**Fixes applied vs original repo:**
- `DATA_PATH` and `OUTPUT_DIR` now default to local paths (no Kaggle dependency)
- Added missing CNN model training section (MobileNetV2 transfer learning)
- Saves `currency_model.h5` and `class_indices.json` for the detector notebook
- Fixed truncated cells (feature distribution plots, box plots, etc.)
- Added `class_indices.json` export so the detector notebook works out-of-the-box
- Added model evaluation: confusion matrix, classification report, per-class accuracy
- Guard against empty DATA_PATH pointing at cwd root

**Sections**
1. Imports & Configuration
2. Data Loading & EDA
3. Data Cleaning & Quality Filtering
4. Handcrafted Feature Extraction & Analysis
5. CNN Model Training (MobileNetV2 Transfer Learning)
6. Model Evaluation
7. Save Artefacts

---
## 1 · Imports & Configuration

In [ ]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ All libraries imported successfully")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit these two paths if needed
# ─────────────────────────────────────────────────────────────
DATA_PATH  = 'datasets'   # FIX: was '' (empty string → scanned entire working dir)
OUTPUT_DIR = 'outputs'    # FIX: was '/kaggle/working/' (Kaggle-only path)

IMG_SIZE   = (224, 224)   # CNN input size
BATCH_SIZE = 32
EPOCHS     = 30           # EarlyStopping will cut this short if val_loss plateaus
SEED       = 42
# ─────────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if not Path(DATA_PATH).is_dir():
    raise FileNotFoundError(
        f"Dataset directory '{DATA_PATH}' not found. "
        "Make sure the 'datasets/' folder (with one sub-folder per currency) "
        "is in the same directory as this notebook."
    )

print(f"Data Path   : {Path(DATA_PATH).resolve()}")
print(f"Output Dir  : {Path(OUTPUT_DIR).resolve()}")

---
## 2 · Data Loading & EDA

In [ ]:
def load_dataset_info(data_path):
    """Walk DATA_PATH and build a DataFrame of (path, currency, filename, size_kb)."""
    records = []
    for img_path in Path(data_path).rglob('*'):
        if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
            records.append({
                'image_path' : str(img_path),
                'currency'   : img_path.parent.name.upper(),
                'filename'   : img_path.name,
                'file_size_kb': img_path.stat().st_size / 1024
            })
    return pd.DataFrame(records)

df_raw = load_dataset_info(DATA_PATH)

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f"Total Images  : {len(df_raw):,}")
print(f"Unique Classes: {df_raw['currency'].nunique()}")
print(f"\nClass Distribution:")
print(df_raw['currency'].value_counts())
print(f"\nFile Size Statistics (KB):")
print(df_raw['file_size_kb'].describe())

In [ ]:
def display_sample_images(df, n_samples=3):
    """Show n_samples images for every currency class."""
    classes  = df['currency'].unique()
    n_classes = len(classes)

    fig, axes = plt.subplots(n_classes, n_samples, figsize=(5 * n_samples, 3 * n_classes))
    # Ensure axes is always 2-D
    if n_classes == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle('Sample Images from Each Currency Class', fontsize=16, y=1.01)

    for i, currency in enumerate(sorted(classes)):
        pool = df[df['currency'] == currency]
        samples = pool.sample(min(n_samples, len(pool)), random_state=SEED)
        for j, (_, row) in enumerate(samples.iterrows()):
            img = cv2.imread(row['image_path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{currency}\n{img.shape[1]}×{img.shape[0]}", fontsize=9)
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=120, bbox_inches='tight')
    plt.show()

display_sample_images(df_raw)

---
## 3 · Data Cleaning & Quality Filtering

In [ ]:
def remove_corrupt_images(df):
    valid_indices, corrupt_count = [], 0
    print("Checking for corrupt images...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Validating'):
        try:
            with Image.open(row['image_path']) as img:
                img.verify()          # PIL structural check
            test = cv2.imread(row['image_path'])
            if test is not None:
                valid_indices.append(idx)
            else:
                corrupt_count += 1
        except Exception as e:
            corrupt_count += 1
            print(f"  Corrupt: {row['filename']} — {str(e)[:60]}")

    print(f"\nValid images  : {len(valid_indices)}")
    print(f"Corrupt removed: {corrupt_count}")
    return df.loc[valid_indices].reset_index(drop=True)

df_clean = remove_corrupt_images(df_raw)

In [ ]:
def extract_image_properties(df):
    """Append width, height, aspect_ratio, total_pixels, channels columns."""
    props = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Reading dims'):
        img = cv2.imread(row['image_path'])
        h, w = img.shape[:2]
        props.append({
            'width'       : w,
            'height'      : h,
            'aspect_ratio': w / h,
            'total_pixels': w * h,
            'channels'    : img.shape[2] if len(img.shape) == 3 else 1
        })
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(props)], axis=1)

df_clean = extract_image_properties(df_clean)
print("Image Property Statistics:")
print(df_clean[['width', 'height', 'aspect_ratio', 'total_pixels']].describe())

In [ ]:
def filter_quality_issues(df, min_pixels=10_000, min_aspect=0.2, max_aspect=5.0):
    initial = len(df)
    df = df[df['total_pixels'] >= min_pixels]
    df = df[(df['aspect_ratio'] >= min_aspect) & (df['aspect_ratio'] <= max_aspect)]
    print(f"Quality Filtering: removed {initial - len(df)} images → {len(df)} remaining")
    return df.reset_index(drop=True)

df_clean = filter_quality_issues(df_clean)

In [ ]:
def analyze_class_balance(df):
    counts = df['currency'].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Currency'); axes[0].set_ylabel('Images')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)

    counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Class Share (%)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()

    ratio = counts.max() / counts.min()
    print(f"Imbalance ratio: {ratio:.2f}")
    if ratio > 3:
        print("⚠️  Significant imbalance — data augmentation recommended")
    return counts

class_distribution = analyze_class_balance(df_clean)

In [ ]:
def remove_rare_classes(df, min_samples=5):
    counts  = df['currency'].value_counts()
    valid   = counts[counts >= min_samples].index
    rare    = counts[counts < min_samples].index
    if len(rare):
        print(f"Removing {len(rare)} rare class(es): {list(rare)}")
        df = df[df['currency'].isin(valid)].reset_index(drop=True)
    else:
        print("No rare classes found ✅")
    return df

# Need at least 5 samples per class to allow stratified splitting across train/val/test
df_clean = remove_rare_classes(df_clean, min_samples=5)
print(f"\nFinal dataset: {len(df_clean):,} images across {df_clean['currency'].nunique()} classes")

---
## 4 · Handcrafted Feature Extraction & Analysis

In [ ]:
def extract_image_features(img_path: str) -> dict:
    """Extract colour, edge, and texture features from a single image."""
    img  = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    feats = {}

    # Brightness
    feats['brightness_mean'] = float(np.mean(gray))
    feats['brightness_std']  = float(np.std(gray))
    feats['contrast']        = float(gray.max() - gray.min())

    # Per-channel colour histograms (BGR order from cv2)
    for i, color in enumerate(['blue', 'green', 'red']):
        hist = cv2.calcHist([img], [i], None, [32], [0, 256]).flatten()
        feats[f'{color}_mean'] = float(np.mean(hist))
        feats[f'{color}_std']  = float(np.std(hist))
        feats[f'{color}_max']  = float(np.max(hist))

    # Edge features
    edges = cv2.Canny(gray, 50, 150)
    feats['edge_density'] = float(np.sum(edges > 0) / edges.size)
    feats['edge_mean']    = float(np.mean(edges))

    # Texture (Laplacian variance)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    feats['texture_variance'] = float(lap.var())

    # Colour dominance
    b, g, r = cv2.split(img)
    total = np.mean(img) + 1e-6
    feats['blue_dominance']  = float(np.mean(b) / total)
    feats['green_dominance'] = float(np.mean(g) / total)
    feats['red_dominance']   = float(np.mean(r) / total)

    return feats

print("Feature extractor defined ✅")

In [ ]:
feature_list = []
for _, row in tqdm(df_clean.iterrows(), total=len(df_clean), desc='Extracting features'):
    feats = extract_image_features(row['image_path'])
    feats['currency']   = row['currency']
    feats['image_path'] = row['image_path']
    feature_list.append(feats)

df_features = pd.DataFrame(feature_list)
feature_cols = [c for c in df_features.columns if c not in {'currency', 'image_path'}]

print(f"Features extracted: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")
df_features[feature_cols].describe()

In [ ]:
# Correlation heatmap
corr = df_features[feature_cols].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Print highly correlated pairs
high = np.where(np.abs(corr) > 0.8)
pairs = [(corr.index[x], corr.columns[y], corr.iloc[x, y])
         for x, y in zip(*high) if x < y]
print("Highly correlated feature pairs (|r| > 0.8):")
for f1, f2, v in pairs:
    print(f"  {f1} ↔ {f2}: {v:.3f}")

In [ ]:
# Feature distribution histograms
key_features = ['brightness_mean', 'contrast', 'edge_density',
                'red_mean', 'green_mean', 'blue_mean']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feat in enumerate(key_features):
    for currency in df_features['currency'].unique():
        data = df_features[df_features['currency'] == currency][feat]
        axes[idx].hist(data, alpha=0.6, label=currency, bins=20)
    axes[idx].set_xlabel(feat.replace('_', ' ').title())
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(feat.replace('_', ' ').title())
    axes[idx].legend(fontsize=7)
    axes[idx].grid(alpha=0.3)

plt.suptitle('Feature Distributions Across Currency Classes', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feat in enumerate(key_features):
    df_features.boxplot(column=feat, by='currency', ax=axes[idx])
    axes[idx].set_xlabel('Currency')
    axes[idx].set_ylabel(feat.replace('_', ' ').title())
    axes[idx].set_title(feat.replace('_', ' ').title())
    axes[idx].get_figure().suptitle('')   # suppress auto-title
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Feature Variance Across Classes (Box Plots)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# PCA visualisation
X = df_features[feature_cols].values
y = df_features['currency'].values

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca   = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
for currency in sorted(df_features['currency'].unique()):
    mask = y == currency
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=currency, alpha=0.6, s=40)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=12)
plt.title('PCA Projection of Currency Notes', fontsize=16, fontweight='bold')
plt.legend(fontsize=8, ncol=2)
plt.grid(alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/pca_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

---
## 5 · CNN Model Training (MobileNetV2 Transfer Learning)

This section was **missing** from the original repo but is required by `Currency_Note_Detector.ipynb`.
We use MobileNetV2 pre-trained on ImageNet with a custom classification head.

In [ ]:
# ── Build file-path splits from df_clean ──────────────────────────────────────
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['currency'])
CLASS_NAMES = list(le.classes_)
NUM_CLASSES = len(CLASS_NAMES)

# Save class indices JSON (format expected by Currency_Note_Detector.ipynb)
class_indices = {name: int(idx) for idx, name in enumerate(CLASS_NAMES)}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print("class_indices.json saved ✅")

In [ ]:
# Stratified 70 / 15 / 15 split on image paths
train_val, test_df = train_test_split(
    df_clean, test_size=0.15, stratify=df_clean['currency'], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val, test_size=0.176, stratify=train_val['currency'], random_state=SEED
    # 0.176 ≈ 15 / 85 → gives ~15% of full set as validation
)

print(f"Train      : {len(train_df):>5} images ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"Validation : {len(val_df):>5} images ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"Test       : {len(test_df):>5} images ({len(test_df)/len(df_clean)*100:.1f}%)")

In [ ]:
# ── ImageDataGenerators ───────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

def df_to_generator(datagen, df, shuffle=False):
    return datagen.flow_from_dataframe(
        dataframe   = df,
        x_col       = 'image_path',
        y_col       = 'currency',
        target_size = IMG_SIZE,
        batch_size  = BATCH_SIZE,
        class_mode  = 'categorical',
        classes     = CLASS_NAMES,
        shuffle     = shuffle,
        seed        = SEED
    )

train_gen = df_to_generator(train_datagen, train_df, shuffle=True)
val_gen   = df_to_generator(val_datagen,   val_df,   shuffle=False)
test_gen  = df_to_generator(test_datagen,  test_df,  shuffle=False)

print(f"Steps per epoch (train): {len(train_gen)}")
print(f"Steps per epoch (val)  : {len(val_gen)}")

In [ ]:
# ── Build model: MobileNetV2 backbone + custom head ───────────────────────────
def build_model(num_classes, img_size=IMG_SIZE):
    base = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(*img_size, 3)
    )
    # Freeze backbone initially
    base.trainable = False

    inputs = keras.Input(shape=(*img_size, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base

model, base_model = build_model(NUM_CLASSES)
model.summary()

In [ ]:
# ── Phase 1: Train head only ──────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('currency_model_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print("=" * 60)
print("PHASE 1: Training classification head (backbone frozen)")
print("=" * 60)

history1 = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1
)

In [ ]:
# ── Phase 2: Fine-tune top layers of backbone ─────────────────────────────────
base_model.trainable = True

# Freeze all but the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreezing: {trainable}")

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),   # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('currency_model_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print("=" * 60)
print("PHASE 2: Fine-tuning top backbone layers")
print("=" * 60)

history2 = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1
)

In [ ]:
# ── Plot combined training curves ─────────────────────────────────────────────
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history = merge_histories(history1, history2)
phase_boundary = len(history1.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history[metric],     label=f'Train {title}', linewidth=2)
    ax.plot(history[f'val_{metric}'], label=f'Val {title}',   linewidth=2)
    ax.axvline(phase_boundary - 1, color='grey', linestyle='--',
               alpha=0.7, label='Fine-tune starts')
    ax.set_title(f'Training {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6 · Model Evaluation

In [ ]:
# Load best checkpoint for evaluation
model.load_weights('currency_model_best.h5')

test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\n{'='*60}")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"{'='*60}")

In [ ]:
# Predictions
test_gen.reset()
y_prob = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_gen.classes

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(max(10, NUM_CLASSES), max(8, NUM_CLASSES - 2)))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.title('Normalised Confusion Matrix', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(14, 5))
bars = plt.bar(CLASS_NAMES, per_class_acc * 100, color='steelblue', edgecolor='white')
plt.axhline(y=test_acc * 100, color='red', linestyle='--', linewidth=1.5,
            label=f'Overall ({test_acc*100:.1f}%)')
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=8)
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Currency'); plt.ylabel('Accuracy (%)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 110)
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7 · Save Artefacts

In [ ]:
# Save final model as currency_model.h5  (name expected by detector notebook)
model.save('currency_model.h5')
print("currency_model.h5 saved ✅")

# Persist class_indices.json in outputs/ as well
import shutil
shutil.copy('class_indices.json', f'{OUTPUT_DIR}/class_indices.json')

# Summary JSON
summary = {
    'analysis_date'      : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_images_raw'   : len(df_raw),
    'total_images_clean' : len(df_clean),
    'num_classes'        : NUM_CLASSES,
    'class_names'        : CLASS_NAMES,
    'train_size'         : len(train_df),
    'val_size'           : len(val_df),
    'test_size'          : len(test_df),
    'test_accuracy'      : round(float(test_acc), 4),
    'test_loss'          : round(float(test_loss), 4),
    'per_class_accuracy' : {n: round(float(a), 4) for n, a in zip(CLASS_NAMES, per_class_acc)}
}
with open(f'{OUTPUT_DIR}/final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\n✅ All artefacts saved successfully!")
print("   → currency_model.h5        (use with Currency_Note_Detector.ipynb)")
print("   → class_indices.json       (use with Currency_Note_Detector.ipynb)")
print(f"  → {OUTPUT_DIR}/            (plots, CSVs, summary JSON)")